# Data Agent 评测教程：以电商问数为例

这本 notebook 把 `evals/` 目录的评测体系从头讲一遍，并用 **glm-5.3-flash 的真实评测结果**做解读。读完你应该能回答：

1. 一个 NL2SQL Agent 的“答对”该怎么定义，为什么比较**执行结果**而不是 SQL 文本
2. 评测题怎么出才有区分度：每道题都有一条“典型错误写法”，并且要证明它真的能被判错
3. 答错时怎么知道错在哪一环：召回、过滤，还是 SQL 写错
4. 怎么用评测做回归门禁和消融实验：改了 Prompt、关掉某个模块，准确率变了多少
5. 多轮对话怎么评：teacher forcing 和自由对话分别衡量什么

**怎么读：** 第 1～5 节只需要 MySQL 里有数仓数据；第 6～7 节会真的让 Agent 答一道题，需要全部服务（MySQL、ES、Qdrant、TEI）、构建好的知识库和 LLM key；第 9～10 节读取已提交的评测结果，不需要任何服务。

| 前提 | 命令 |
| --- | --- |
| 安装依赖 | `uv sync --group notebook` |
| 启动服务 | `docker compose -f docker/docker-compose.yaml up -d` |
| 写入数仓数据 | `uv run python -m app.scripts.generate_dw_data`（或导入 `docker/snapshot/dw_snapshot.sql.gz`） |
| 构建知识库 | `uv run python -m app.scripts.build_meta_knowledge -c conf/meta_config.yaml` |
| LLM key | 项目根目录 `.env` 里写 `LLM_API_KEY=...`（智谱开放平台） |

Windows + Docker 与 Linux 的差异见文末附录。

## 0. 准备

In [ ]:
import inspect
import json
import os
import sys
from pathlib import Path

import pandas as pd
import yaml

# 项目根目录：notebook 在 notebooks/ 下运行时取上一级
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # 配置里的日志目录等相对路径都以项目根目录为准
sys.path.insert(0, str(ROOT))

# 导入项目代码前先把控制台日志调到 WARNING，否则每个节点的 INFO 日志会刷屏
from app.conf.app_config import app_config  # noqa: E402

app_config.logging.console.level = "WARNING"

from evals.compare import results_equal  # noqa: E402
from evals.dataset import (  # noqa: E402
    DEFAULT_DATASET,
    MULTITURN_DATASET,
    check_leakage,
    check_structure,
    load_dataset,
    load_examples,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

traps = load_dataset(DEFAULT_DATASET)
multiturn = load_dataset(MULTITURN_DATASET)
print(f"单轮题 {len(traps)} 道，多轮对话 {len(multiturn)} 段")

## 1. 评测要回答什么

一次评测要回答四个问题：

| 问题 | 指标 | 代码 |
| --- | --- | --- |
| 答对了吗 | 执行准确率（EX）：模型 SQL 与标准 SQL 的执行结果等价的比例 | `evals/compare.py` |
| 错在哪 | 失败环节：流程异常 → 召回遗漏 → 过滤误删 → SQL 不可执行 → 结果不符（多轮再加“意图错误”） | `evals/scoring.py` |
| 代价多大 | 每题 LLM 调用次数、token 数、耗时、SQL 修正次数 | `evals/runner.py` |
| 变差了吗 | 与基线比较：必过题失败、准确率下降超过阈值、成本上升超过阈值 | `scoring.compare_with_baseline` |

整体流程：

```
评测集 YAML ──► 答题 (predictor: agent / gold / trap)
                  │  agent：跑问数图，捕获每个节点的输出
                  ▼
        执行预测 SQL 与标准 SQL ──► 判分 results_equal ──► 归因 attribute_failure
                                                              │
                         results.jsonl / summary.json / summary.md ◄── 汇总 summarize
                                                              │
                                                  与基线比较（退化时退出码 1）
```

文件地图：

| 文件 | 职责 | 需要数据库 |
| --- | --- | --- |
| `evals/datasets/traps.yaml` | 单轮题：10 类业务口径陷阱 × 5 题 | |
| `evals/datasets/multiturn.yaml` | 多轮题：6 类追问 × 20 段对话 | |
| `evals/dataset.py` | 静态检查：字段齐全、expect 与 SQL 一致、只读、与 few-shot 示例不泄漏（pytest 会跑） | 否 |
| `evals/check_dataset.py` | 在库检查：标准 SQL 有结果、错误写法的结果与标准答案不同 | 是 |
| `evals/compare.py` | 结果等价判定（纯函数） | 否 |
| `evals/scoring.py` | 归因、汇总、基线比较、报告（纯函数） | 否 |
| `evals/runner.py` | 单轮答题与打分的命令行入口 | 是 |
| `evals/multiturn.py` | 多轮答题（teacher / free） | 是 |
| `evals/llm_cache.py` | LLM 调用录制回放（SQLite） | 否 |

## 2. 评测集：题目长什么样

这套题不考“会不会写 SQL”，考的是**业务口径**：每道题都对应一种“SQL 能跑、结果却错了”的陷阱，比如把未支付订单算进销售额、订单头金额关联明细后重复累加、把收货地当成用户常住地。

每道题的字段：

| 字段 | 作用 |
| --- | --- |
| `question` / `paraphrases` | 问题和其他问法（测“换个说法还能不能答对”） |
| `gold_sql` | 标准 SQL，判分时比较它的执行结果 |
| `trap_sql` / `trap` | 这类陷阱的典型错误写法和错在哪。它让题目“可证伪”：如果错误写法也能得到同样结果，这道题就测不出这个陷阱 |
| `expect` | 回答本题需要的表 / 指标 / 取值，用于答错时判断是不是召回或过滤阶段漏了 |
| `order_matters` | 只有问题本身要求排序时才比较行顺序 |
| `today` | 注入给 Agent 的“今天”，让“本月”“去年同期”可复现 |
| `must_pass` | 核心题，失败即视为回归 |

In [ ]:
summary = pd.DataFrame(
    [
        {
            "类别": item["category"],
            "题目": item["id"],
            "难度": item["difficulty"],
            "必过": bool(item.get("must_pass")),
            "问题": item["turns"][0]["question"],
        }
        for item in traps
    ]
)
summary.groupby("类别").agg(
    题数=("题目", "count"), 必过=("必过", "sum"), 示例=("问题", "first")
)

看一道完整的题：`fanout_01` 考的是“订单头金额关联明细后求和会被重复计算”。

In [ ]:
item = next(i for i in traps if i["id"] == "fanout_01")
print(yaml.dump(item, allow_unicode=True, sort_keys=False))

## 3. 为什么比较执行结果，而不是 SQL 文本

同一个问题可以有很多种正确写法：按支付日期筛时间，可以直接写 `pay_date_id BETWEEN ...`，也可以关联 `dim_date` 按 `year` 筛；列名可以叫“销售额”也可以叫 `gmv`。逐字比较 SQL 会把正确答案判错。

所以主流的 NL2SQL 基准（Spider、BIRD）都用**执行准确率（Execution Accuracy, EX）**：两条 SQL 在同一个库上执行，结果等价就算对。代价是需要一个固定的数据库，并且要定义清楚“结果等价”（下一节）。

下面在数仓上验证：两种正确写法结果相同，错误写法结果不同。

In [ ]:
from app.clients.mysql_client_manager import dw_mysql_client_manager
from evals.runner import execute  # runner 用的同一个执行函数：只读校验后执行

dw_mysql_client_manager.init()

gold = item["turns"][0]["gold_sql"]
# 另一种正确写法：关联日期维度按年筛选，列名也不同
variant = """
select c.category_l1 as cat, sum(i.pay_amount) as gmv
from fact_order_item i
join fact_order o on i.order_id = o.order_id
join dim_date d on o.pay_date_id = d.date_id
join dim_sku k on i.sku_id = k.sku_id
join dim_category c on k.category_id = c.category_id
where o.pay_time is not null and d.year = 2025
group by c.category_l1
"""
trap = item["turns"][0]["trap_sql"]

gold_rows = await execute(gold)
variant_rows = await execute(variant)
trap_rows = await execute(trap)
print("另一种正确写法判为相同：", results_equal(variant_rows, gold_rows))
print("错误写法判为相同：", results_equal(trap_rows, gold_rows))
pd.DataFrame(gold_rows).merge(
    pd.DataFrame(trap_rows), on="一级类目", suffixes=("_标准", "_错误写法")
)

## 4. 判分规则逐条拆解（`evals/compare.py`）

`results_equal(pred, gold, order_matters)` 判断两个“列名 → 值”的字典列表是否等价，规则参照 Spider 2.0 的宽松口径：

1. **忽略列名和列顺序**：标准答案的每一列，都要在预测结果里找到一列内容相同的列（列名是模型起的，不该影响对错）
2. **允许多出列**：问“哪个大区最高”，模型顺手带上销售额也算对；但不能少列、行数必须一致
3. **行顺序**：只有 `order_matters` 时才逐行比较，否则当多重集合比较
4. **数值容差**：相对误差 1e-9（只吸收浮点误差，金额动辄几百万，放宽会放过几元的差异）；预测值四舍五入到 1～4 位小数也算对；同一列整体是标准值的 100 倍也算对（比率 vs 百分数）
5. **类型归一**：Decimal、整数统一转 float，日期转 ISO 字符串，字符串去掉首尾空白

实现上分两步：先为每个标准列找出所有“内容能对上”的预测列（含倍数），再回溯搜索一组互不重复、并且逐行也能对齐的对应关系。第二步是必要的：两列各自都能对上，不代表它们在同一行上是配对的。

In [ ]:
G = [{"area": "华东", "amt": 1234.5678}, {"area": "华南", "amt": 999.1}]
cases = [
    (
        "列名不同",
        [{"大区": "华东", "销售额": 1234.5678}, {"大区": "华南", "销售额": 999.1}],
        False,
    ),
    (
        "列顺序不同",
        [{"amt": 1234.5678, "area": "华东"}, {"amt": 999.1, "area": "华南"}],
        False,
    ),
    (
        "多出一列",
        [
            {"area": "华东", "amt": 1234.5678, "n": 3},
            {"area": "华南", "amt": 999.1, "n": 1},
        ],
        False,
    ),
    ("少一列", [{"area": "华东"}, {"area": "华南"}], False),
    (
        "四舍五入到 2 位",
        [{"area": "华东", "amt": 1234.57}, {"area": "华南", "amt": 999.1}],
        False,
    ),
    (
        "四舍五入但值错了",
        [{"area": "华东", "amt": 1234.58}, {"area": "华南", "amt": 999.1}],
        False,
    ),
    (
        "差 1 分钱",
        [{"area": "华东", "amt": 1234.5778}, {"area": "华南", "amt": 999.1}],
        False,
    ),
    (
        "行顺序不同（不要求顺序）",
        [{"area": "华南", "amt": 999.1}, {"area": "华东", "amt": 1234.5678}],
        False,
    ),
    (
        "行顺序不同（要求顺序）",
        [{"area": "华南", "amt": 999.1}, {"area": "华东", "amt": 1234.5678}],
        True,
    ),
    (
        "字符串带空格",
        [{"area": " 华东 ", "amt": 1234.5678}, {"area": "华南", "amt": 999.1}],
        False,
    ),
    (
        "行列错配（列都能对上，行对不上）",
        [{"area": "华东", "amt": 999.1}, {"area": "华南", "amt": 1234.5678}],
        False,
    ),
]
rows = [
    {
        "情形": name,
        "order_matters": om,
        "判定": "相等" if results_equal(pred, G, om) else "不等",
    }
    for name, pred, om in cases
]
# 比率 vs 百分数
rows.append(
    {
        "情形": "比率写成百分数（×100）",
        "order_matters": False,
        "判定": "相等" if results_equal([{"r": 12.5}], [{"rate": 0.125}]) else "不等",
    }
)
pd.DataFrame(rows)

**宽松是有代价的**：规则越宽，越可能把错误答案判对。例如错误写法的结果恰好是标准答案的 100 倍、或者四舍五入后恰好相同。所以第 5 节的自检要求：每道题的错误写法，在“保留两位小数”之后仍然要能和标准答案区分开。

## 5. 评测集自检：测不出错的题没有意义

一道题如果错误写法也能“答对”，它就测不出它声称的陷阱。自检分两层：

- **静态检查**（`evals/dataset.py`，pytest 每次都跑）：必填字段齐全、id 唯一、每类题数、`expect.tables` 都出现在标准 SQL 里、`expect.metrics` 都是已定义的指标、两条 SQL 都能通过只读校验，以及**防泄漏**：few-shot 示例和评测题的问题相似度 ≥ 0.8、或 SQL 相似度 ≥ 0.9 就报错（示例会被召回进 Prompt，太像评测题分数就会虚高）
- **在库检查**（`evals/check_dataset.py`）：标准 SQL 结果非空；错误写法能执行，结果与标准答案不同，并且四舍五入到两位小数后仍然不同

改了数据生成器、数据规模或题目后都要重跑在库检查：`uv run python -m evals.check_dataset`

In [ ]:
print("单轮静态检查问题：", check_structure(traps, per_category=5))
print("多轮静态检查问题：", check_structure(multiturn, per_category=None))
print("示例泄漏：", check_leakage(traps + multiturn, load_examples()))

在库检查的核心逻辑，对每类挑一题演示（完整版见 `check_dataset.check_results`）：

In [ ]:
from evals.check_dataset import _rounded

rows = []
for category in summary["类别"].unique():
    it = next(i for i in traps if i["category"] == category)
    turn = it["turns"][0]
    g = await execute(turn["gold_sql"])
    t = await execute(turn["trap_sql"])
    om = bool(turn.get("order_matters"))
    rows.append(
        {
            "题目": it["id"],
            "标准结果行数": len(g),
            "错误写法与标准相同": results_equal(t, g, om),
            "两位小数后相同": results_equal(_rounded(t), g, om),
            "陷阱": turn["trap"],
        }
    )
pd.DataFrame(rows)

## 6. 让 Agent 答一道题，看它每一步做了什么

`--predictor agent` 时，runner 调用 `run_agent(question, today)`：

- 用 `graph.astream(stream_mode=["updates", "custom"])` 跑问数图。`updates` 模式会在每个节点执行完后给出它返回的 dict，runner 把它们按节点名存下来，**归因就靠这些中间结果**
- 通过 LangChain 回调 `UsageCallback` 统计 LLM 调用次数和 token
- 最终 SQL 取 `correct_sql` 的输出（修正过），没有修正过就取 `generate_sql` 的输出

另外两种 predictor 用来**检验评测系统本身**：`gold` 直接用标准 SQL 作答，必须是 100%；`trap` 用错误写法作答，必须是 0%。任何一个不符合，说明判分或题目有问题，先修评测再看模型。

下面这个单元会真正调用 LLM（约 6 次）。设置了和评测相同的 LLM 缓存：如果评测时录制过这道题，这里会直接回放，不花钱、结果也和评测一致。

In [ ]:
from langchain_core.globals import set_llm_cache

from app.clients.embedding_client_manager import embedding_client_manager
from app.clients.es_client_manager import es_client_manager
from app.clients.mysql_client_manager import meta_mysql_client_manager
from app.clients.qdrant_client_manager import qdrant_client_manager
from evals.llm_cache import SQLiteLLMCache
from evals.runner import available_items, run_agent
from evals.scoring import missing_items

for client in (
    meta_mysql_client_manager,
    qdrant_client_manager,
    es_client_manager,
    embedding_client_manager,
):
    client.init()
cache = SQLiteLLMCache(ROOT / "evals" / ".cache" / "glm-5.3-flash.sqlite")
set_llm_cache(cache)

item = next(i for i in traps if i["id"] == "region_role_01")
turn = item["turns"][0]
answer = await run_agent(turn["question"], item["today"])
print("问题：", turn["question"])
print(
    f"LLM 调用 {answer['llm_calls']} 次（缓存命中 {cache.hits}，未命中 {cache.misses}），耗时 {answer['seconds']:.1f} 秒"
)
print("经过的节点：", list(answer["outputs"]))
print("\n模型 SQL：\n" + answer["sql"])
print("\n标准 SQL：\n" + turn["gold_sql"])

In [ ]:
recalled, filtered = available_items(answer["outputs"])
expect = turn["expect"]
pd.DataFrame(
    {
        "期望": {k: sorted(expect.get(k, [])) for k in ("tables", "metrics", "values")},
        "召回到": {
            k: sorted(recalled.get(k, set())) for k in ("tables", "metrics", "values")
        },
        "过滤后": {
            k: sorted(filtered.get(k, set())) for k in ("tables", "metrics", "values")
        },
    }
)

In [ ]:
pred_rows = await execute(answer["sql"])
gold_rows = await execute(turn["gold_sql"])
print("召回遗漏：", missing_items(expect, recalled))
print(
    "过滤误删：",
    missing_items({k: v for k, v in expect.items() if k != "values"}, filtered),
)
print(
    "结果等价：", results_equal(pred_rows, gold_rows, bool(turn.get("order_matters")))
)

## 7. 失败归因：错在哪一环

答错时，按流水线顺序找**第一个**出问题的环节（借鉴 Uber QueryGPT 按意图、选表、生成分别统计的做法）：

In [ ]:
from evals import scoring

print(inspect.getsource(scoring.attribute_failure))

为什么按这个顺序：

- **召回遗漏**排在最前：需要的表或指标在召回阶段就没拿到，后面的过滤和生成不可能补回来，这时优化 Prompt 是白费力气
- **过滤误删**：召回到了，但 `filter_table` / `filter_metric` 把它删了
- **SQL 不可执行**：上下文齐全，但 SQL 修正次数用完仍然报错
- **结果不符**：能执行但结果错，通常是口径或逻辑错误（例如用了下单口径、关联出扇出）

几个需要注意的局限：

- 召回和过滤的判断完全依赖题目里人工标注的 `expect`，标注不全就会把问题归到后面的环节
- 取值（`values`）只检查召回、不检查过滤，因为过滤节点只输出表和字段
- “结果不符”是兜底类别，里面混着各种逻辑错误，需要看失败明细里的 SQL 才能细分
- 多轮评测在最前面再加一个“意图错误”：意图判错且答错的轮次优先归到这里

归因的价值在于指导优化方向：如果大部分失败是召回遗漏，就该优化检索词扩展、阈值或别名，而不是去调 SQL 生成的 Prompt。

## 8. 跑一次完整评测

完整评测耗时较长（单轮 50 题约 15 分钟），建议在终端里跑，notebook 只用来读结果。以下命令以 Linux / macOS 的 bash 为准，Windows 的写法见附录。

```bash
uv run python -m evals.runner --predictor gold        # 自检：必须 100%
uv run python -m evals.runner --predictor trap        # 自检：必须 0%
uv run python -m evals.runner --paraphrases --llm-cache evals/.cache/glm-5.3-flash.sqlite \
    --baseline evals/baselines/glm-5.3-flash.json --save-baseline       # 基线（含换问法）
uv run python -m evals.runner --no-examples --llm-cache evals/.cache/glm-5.3-flash.sqlite        # 消融：去掉 few-shot 示例
uv run python -m evals.runner --no-semantic-check --llm-cache evals/.cache/glm-5.3-flash.sqlite  # 消融：去掉语义检查
uv run python -m evals.runner --baseline evals/baselines/glm-5.3-flash.json                       # 与基线比较，退化时退出码 1
uv run python -m evals.runner --multiturn teacher --llm-cache evals/.cache/glm-5.3-flash.sqlite  # 多轮
```

**LLM 缓存**（`evals/llm_cache.py`）：以“模型参数 + 完整提示词”为键，把模型回答存进 SQLite。重跑时提示词没变就直接回放，不花钱、结果完全可复现；只要提示词有一个字变了（改了 Prompt、知识库、召回结果），对应的调用就会真实请求一次。做消融实验时，没受影响的调用都能命中缓存，成本很低。

**基线与回归门禁**（`scoring.compare_with_baseline`）：判为退化的条件是任一必过题失败、整体准确率下降超过 2 个百分点（`--max-accuracy-drop`），或平均 LLM 调用次数上升超过 30%。退化时退出码为 1，可以直接接进 CI。

**报告**写到 `evals/reports/<时间戳>/`（已 gitignore）：`results.jsonl` 每题一行明细（SQL、错误、归因、成本），`summary.json` 汇总，`summary.md` 可读报告。本仓库把 glm-5.3-flash 的一次完整评测结果复制到了 `evals/results/glm-5.3-flash/`，下面两节读的就是它。

## 9. 解读真实结果（glm-5.3-flash，单轮）

`RESULTS` 指向已提交的结果；想读自己跑出来的结果，把它改成 `evals/reports/<时间戳>` 所在的目录并按同样的子目录名放好即可。

In [ ]:
RESULTS = ROOT / "evals" / "results" / "glm-5.3-flash"


def load_run(name):
    """读取一次评测的汇总和逐题明细"""
    folder = RESULTS / name
    s = json.loads((folder / "summary.json").read_text(encoding="utf-8"))
    r = pd.DataFrame(
        [
            json.loads(line)
            for line in (folder / "results.jsonl")
            .read_text(encoding="utf-8")
            .splitlines()
        ]
    )
    return s, r


runs = {name: load_run(name) for name in ("baseline", "no_examples", "no_lint")}
pd.DataFrame(
    {
        name: {
            "题数": s["total"],
            "准确率": f"{s['accuracy']:.1%}",
            "平均 LLM 调用": s["avg_llm_calls"],
            "平均 tokens": s["avg_tokens"],
            "平均耗时(秒)": s["avg_seconds"],
            "必过题失败": len(s["must_pass_failed"]),
        }
        for name, (s, _) in runs.items()
    }
)

- `baseline`：完整配置，每题原问法 + 一种换问法，共 100 次作答
- `no_examples`：关掉已验证问答示例（few-shot），只答原问法
- `no_lint`：关掉 SQL 语义检查（扇出、关联条件不符合外键），只答原问法

三者的题数不同，下面比较消融效果时只取原问法（`variant == "question"`）。

In [ ]:
base_s, base_r = runs["baseline"]
q = base_r[base_r["variant"] == "question"]
by_cat = (
    base_r.assign(
        问法=base_r["variant"].map(lambda v: "原问法" if v == "question" else "换问法")
    )
    .pivot_table(index="category", columns="问法", values="passed", aggfunc="mean")
    .mul(100)
    .round(0)
)
for name in ("no_examples", "no_lint"):
    _, r = runs[name]
    by_cat[name] = r.groupby("category")["passed"].mean().mul(100).round(0)
by_cat.loc["整体"] = [
    q["passed"].mean() * 100,
    base_r[base_r["variant"] != "question"]["passed"].mean() * 100,
    runs["no_examples"][1]["passed"].mean() * 100,
    runs["no_lint"][1]["passed"].mean() * 100,
]
by_cat.round(0)

> （单轮总体：跑完真实评测后补充解读）

### 失败出在哪一环

In [ ]:
base_r[~base_r["passed"]].groupby(["stage"]).size().rename("次数").to_frame()

In [ ]:
fails = base_r[~base_r["passed"]][
    [
        "case_id",
        "variant",
        "stage",
        "recall_missing",
        "filter_missing",
        "correct_attempts",
        "error",
    ]
]
fails.sort_values(["stage", "case_id"])

> （失败环节：跑完真实评测后补充解读）

### 消融：去掉一个模块，哪些题由对变错

In [ ]:
def flips(name):
    """与基线原问法相比，消融后由对变错、由错变对的题"""
    _, r = runs[name]
    merged = (
        q.set_index("case_id")["passed"]
        .to_frame("基线")
        .join(r.set_index("case_id")["passed"].rename(name))
    )
    return {
        "由对变错": sorted(merged.index[merged["基线"] & ~merged[name]]),
        "由错变对": sorted(merged.index[~merged["基线"] & merged[name]]),
    }


pd.DataFrame({name: flips(name) for name in ("no_examples", "no_lint")}).T

> （消融：跑完真实评测后补充解读）

### 看几道失败题的 SQL

In [ ]:
gold_by_id = {i["id"]: i["turns"][0]["gold_sql"].strip() for i in traps}
trap_by_id = {i["id"]: i["turns"][0]["trap"] for i in traps}
for _, row in q[~q["passed"]].head(4).iterrows():
    print(f"=== {row['case_id']}（{row['stage']}）{row['question']}")
    print(f"陷阱：{trap_by_id[row['case_id']]}")
    print("--- 模型 SQL\n" + (row["sql"] or f"（无 SQL）{row['error']}"))
    print("--- 标准 SQL\n" + gold_by_id[row["case_id"]] + "\n")

## 10. 多轮评测

多轮题的每一轮都有标准 SQL，第 2 轮起还标注了期望意图（`gold_intent`）和期望的改写结果（`gold_standalone`）。两种模式：

| 模式 | 历史从哪来 | 衡量什么 |
| --- | --- | --- |
| `teacher`（teacher forcing） | 每轮新开会话，历史用前几轮的**标准答案**填充（标准改写、标准 SQL 及其结果预览） | 单轮追问能力：前一轮答错不会连累后面，每一轮都是公平的考题 |
| `free`（自由对话） | 同一个会话依次作答，历史是**模型自己的** | 真实体验：前一轮的错误会传递到后面 |

指标：意图准确率（第 2 轮起）、追问轮准确率、全部轮次准确率、整段对话成功率（每一轮都对）。多轮的归因在单轮之前多一个“意图错误”。

现阶段单轮流水线只接收改写后的问题，所以“改写准确率”就等于轮次准确率；失败明细里会并列预测改写和标准改写，方便判断是改写错还是 SQL 错。

In [ ]:
mt = {name: load_run(name) for name in ("mt_teacher", "mt_free")}
pd.DataFrame(
    {
        name: {
            "整段对话成功率": f"{s['dialog_success_rate']:.0%}（{s['dialog_success']}/{s['dialogs']}）",
            "追问轮准确率": f"{s['followup_accuracy']:.0%}（{s['followup_passed']}/{s['followup_total']}）",
            "全部轮次准确率": f"{s['accuracy']:.0%}",
            "意图准确率": f"{s['intent_accuracy']:.0%}"
            if s["intent_accuracy"] is not None
            else "-",
            "意图混淆": s["intent_confusion"] or "无",
            "每轮平均 LLM 调用": s["avg_llm_calls"],
        }
        for name, (s, _) in mt.items()
    }
)

In [ ]:
pd.DataFrame(
    {
        name: {
            cat: f"{v['followups_passed']}/{v['followups']}"
            for cat, v in s["by_category"].items()
        }
        for name, (s, _) in mt.items()
    }
).rename_axis("追问类别")

In [ ]:
_, mt_r = mt["mt_free"]
mt_r[~mt_r["passed"]][
    [
        "case_id",
        "turn",
        "stage",
        "gold_intent",
        "intent",
        "standalone",
        "gold_standalone",
    ]
]

> （多轮：跑完真实评测后补充解读）

## 11. 动手实验建议

按“改一处、看指标怎么变”的思路，每个实验都有明确要回答的问题：

1. **问法鲁棒性**：比较原问法和换问法的准确率（第 9 节已有）。差距大说明模型依赖特定措辞，可以加别名或业务术语
2. **few-shot 的真实贡献**：`--no-examples` 已经做了；进一步可以只删掉某一条示例，看哪些题受影响，判断示例是在教“写法”还是在“泄题”
3. **召回阈值**：改 `conf/app_config.yaml` 的 `qdrant.score_threshold`（0.6 → 0.5 / 0.7），看召回遗漏和 token 数怎么变
4. **换模型**：改 `llm.model_name`，用同一份评测集比较准确率和成本；用 `--baseline` 看哪些题由对变错
5. **加一道题**：在 `traps.yaml` 加题，先跑 `check_dataset` 证明错误写法可区分，再跑 `--predictor gold` / `trap` 自检
6. **判分规则的边界**：在第 4 节的单元里构造更多用例，想想“允许多出列”会不会放过错误答案

每次实验都加 `--llm-cache`，没变化的调用直接回放；用 `--baseline` 对照基线，退化时一眼就能看出来。

## 附录：Windows + Docker 与 Linux 的差异

本仓库的命令和这本 notebook 都以 Linux（bash）为准，在 Linux 云环境验证过。在 Windows + Docker Desktop 上复现时注意：

| 项目 | Linux（本仓库） | Windows + Docker Desktop |
| --- | --- | --- |
| 命令行里临时设环境变量 | `LLM_API_KEY=xxx uv run ...` | PowerShell：`$env:LLM_API_KEY="xxx"`；最省事是写进项目根目录的 `.env`，代码启动时会自动读取 |
| 导入 SQL 快照 | `gunzip -c docker/snapshot/dw_snapshot.sql.gz \| docker exec -i mysql mysql -uroot -pdili123` | PowerShell 的管道和 `<` 重定向不适合二进制文件，可在 Git Bash / WSL 里执行同一条命令，或先 `docker cp docker/snapshot/dw_snapshot.sql.gz mysql:/tmp/`，再 `docker exec mysql sh -c "gunzip -c /tmp/dw_snapshot.sql.gz \| mysql -uroot -pdili123"` |
| 输出重定向到文件 | 无需处理 | Windows 默认 GBK 编码，runner 输出里的 ✓ ✗ 写入文件时可能报 `UnicodeEncodeError`，先设 `$env:PYTHONUTF8="1"` |
| TEI 镜像 | `linux/amd64` 原生运行 | x64 机器上一样；ARM 机器会走模拟，非常慢 |
| TEI 并发上限 | compose 里 `MAX_CONCURRENT_REQUESTS=512` | 同一个镜像，同样需要（原来的 16 会让知识库构建时 TEI panic 重启） |
| 内存 | 全套服务实测约 6～8 GB（TEI 的 bge-large 约 2～3 GB，ES 约 1～2 GB） | Docker Desktop（WSL2）默认内存可能不够，在 `.wslconfig` 里调到 8 GB 以上 |
| 3306 端口 | 一般空闲 | 本机装过 MySQL 服务时会冲突，先停掉本机服务或改 compose 端口映射 |
| 数据卷初始化 | MySQL 账号由 compose 的 `MYSQL_USER` 在首次初始化时创建 | 同上；改过账号要 `docker compose -f docker/docker-compose.yaml down -v` 重建 |
| 换行符 | 仓库统一 LF | Git 的 `core.autocrlf=true` 会把文件检出成 CRLF。项目读取 SQL、Prompt、YAML 时都按行处理，不受影响 |
| 文件编码 | — | 项目代码读写文件都显式用了 UTF-8，不依赖系统默认编码 |
| 异步事件循环 | 默认的 selector 循环 | Windows 默认是 Proactor 循环，asyncmy、aiohttp 一般可用，**未在 Windows 上实测** |
| 结果可复现性 | LLM `temperature=0`，加上缓存回放后完全可复现 | 不同机器上 TEI 的浮点计算可能有微小差异，召回结果的顺序可能不同，导致提示词变化、缓存不命中、结果略有出入 |